# LookbackLens baseline

([arXiv:2407.07071](https://arxiv.org/abs/2407.07071)).

**Approach**

1. **Forward-pass attention extraction.** For each record, build the prompt `Question: {query}\nContext: {context}\nAnswer: ` and concatenate the gold `output`. Run a frozen causal LM (`Llama-2-7b-chat`, the paper's exact backbone, via the `NousResearch` ungated mirror) in teacher-forced mode with `output_attentions=True`. For every answer-token position `t` and every (layer `l`, head `h`) compute the **lookback ratio**
   $$ \mathrm{LR}_{l,h,t} = \frac{A^{\text{context}}_{l,h,t}}{A^{\text{context}}_{l,h,t} + A^{\text{new}}_{l,h,t}}, $$
   where $A^{\text{context}}$ is the mean attention from `t` over tokens of the tool response and $A^{\text{new}}$ is the mean attention from `t` over previously generated answer tokens. Instruction tokens are intentionally excluded from both averages (paper convention).

2. **Window aggregation.** Average per-token features over disjoint windows of `WINDOW_SIZE` answer tokens (paper uses 4–8). Each window gets a feature vector in $\mathbb{R}^{L \cdot H}$ (= 32 × 32 = 1024 for Llama-2-7B-chat) and a binary label (`1` iff any token in the window overlaps a gold hallucination span).

3. **Logistic-regression classifier.** Fit a balanced-class-weight logistic regression on standardised window features; predict per window; collapse contiguous positive windows into char spans. Two training-data variants × two window sizes × three thresholds = **12 method rows** reported side-by-side

**Inputs/outputs (matches the task PDF):**
- `query` → question, `context` → tool output, `output` → answer; predictions are over char offsets of `output`

**Dataset and metrics:**
- Same 4 toolace configs as the LettuceDetect baseline: `combined`, `contradiction`, `missing_tool`, `overgeneration` (test split)
- Two reporting metrics
  - **Example-level** binary detection: any hallucination in the record?
  - **Span-level** RAGTruth character-overlap micro-P/R/F1

**Two training-data variants:**
- **`lookbacklens-ragtruth-*` (zero-shot transfer).** Classifier trained on attention features from a 5K records sample of `wandb/RAGTruth-processed` — the same training corpus the published `KRLabsOrg/lettucedect-*` checkpoints used. Evaluation on toolace is **zero-shot domain transfer** — directly comparable to LettuceDetect setup
- **`lookbacklens-toolace-*` (in-domain, reference).** Classifier trained on `combined/train` of toolace. Reported alongside as an oracle: same architecture, same backbone, but trained on the target distribution

**Caveat.** The LookbackLens paper trains its classifier on attention from the *same* model that generated the answer. Our `output` strings come from third parties (ToolACE / corruptor / RAGTruth's model fleet), not Llama-2-7B-chat exclusively. So this is more precisely a *probe* of Llama-2-7B's grounding signal on a third-party answer


## 1. Setup

In [ ]:
import json
import re
import sys
import time
import html
import random
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import HfApi
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__}  device={DEVICE}")

DATASET_REPO = "Ivan1008/toolace-hallucination-spans"
CONFIGS = ["combined", "contradiction", "missing_tool", "overgeneration"]
TRAIN_SOURCE_CONFIG = "combined"   

MODEL_NAME = "NousResearch/Llama-2-7b-chat-hf"
MAX_LEN = 4096                       

WINDOW_SIZES = [4, 8]                 # disjoint answer-token window sizes (paper uses 4-8)
THRESHOLDS = [0.3, 0.5, 0.7]          # per-window P(hallucinated) thresholds

CLF_C = 1.0
CLASS_WEIGHT = "balanced"

MAX_TRAIN_RECORDS = None 
MAX_TEST_RECORDS = None
RAGTRUTH_TRAIN_RECORDS = 5000     

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUTPUT_DIR = ROOT / "notebooks" / "results" / "lookbacklens_baseline"
FEATURE_CACHE = OUTPUT_DIR / "features"
RAW_DIR = OUTPUT_DIR / "raw"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_CACHE.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

## Lookback-ratio feature extraction

* `RecordFeatures` - one record per-token features
* `extract_features` - one teacher-forced forward pass through the LLM
* `aggregate_window_features` - per-window averaging

In [ ]:
@dataclass
class RecordFeatures:
    """Per-answer-token lookback-ratio features and gold labels for one record."""
    record_id: str
    corruption_type: str
    output: str
    token_offsets: np.ndarray  
    features: np.ndarray       
    token_labels: np.ndarray   
    gold_spans: list[dict]
    truncated: bool


PROMPT_TEMPLATE = "Question: {query}\nContext: {context}\nAnswer: "


def build_prompt(query: str, context: str) -> tuple[str, int, int]:
    """Return the prompt string and the (start, end) char offsets of `context`."""
    prefix = PROMPT_TEMPLATE.format(query=query, context=context)
    ctx_start = prefix.index(context)
    ctx_end = ctx_start + len(context)
    return prefix, ctx_start, ctx_end


def _char_range_to_tok_range(offsets, char_start, char_end):
    """Return (tok_start, tok_end) such that offsets[tok_start:tok_end] covers
    [char_start, char_end). Uses any-overlap semantics."""
    tok_start = tok_end = None
    for i, (a, b) in enumerate(offsets):
        if a == b:               
            continue
        if b > char_start and a < char_end:
            if tok_start is None:
                tok_start = i
            tok_end = i + 1
    if tok_start is None:
        return 0, 0
    return tok_start, tok_end


def _token_labels_from_spans(offsets, output, gold_spans):
    """Mark each answer token as 1 iff its char range overlaps any gold span."""
    n = offsets.shape[0]
    labels = np.zeros(n, dtype=np.uint8)
    for s in gold_spans:
        gs = max(0, int(s.get("start", 0)))
        ge = min(len(output), int(s.get("end", 0)))
        if ge <= gs:
            continue
        overlap = (offsets[:, 1] > gs) & (offsets[:, 0] < ge) & (offsets[:, 1] > offsets[:, 0])
        labels[overlap] = 1
    return labels


def load_backbone(model_name=MODEL_NAME, device=DEVICE, dtype=torch.bfloat16):
    """Load an attention-output-capable causal LM and its tokenizer."""
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        attn_implementation="eager",  
    ).to(device)
    model.eval()
    return model, tok


@torch.no_grad()
def extract_features(record, model, tokenizer, *, max_length=MAX_LEN, device=DEVICE):
    """Run a teacher-forced forward pass and return per-answer-token lookback features.

    Returns None if the answer ends up with zero tokens after tokenization."""
    query, context, output = record["query"], record["context"], record["output"]
    prefix, ctx_start_char, ctx_end_char = build_prompt(query, context)
    full_text = prefix + output
    ans_start_char = len(prefix)
    ans_end_char = len(full_text)

    enc = tokenizer(full_text, return_offsets_mapping=True, add_special_tokens=False,
                    truncation=True, max_length=max_length, return_tensors=None)
    input_ids = torch.tensor([enc["input_ids"]], device=device)
    offsets = enc["offset_mapping"]
    seq_len = input_ids.shape[1]
    truncated = enc.get("length", seq_len) > max_length or seq_len == max_length

    ctx_tok_start, ctx_tok_end = _char_range_to_tok_range(offsets, ctx_start_char, ctx_end_char)
    ans_tok_start, ans_tok_end = _char_range_to_tok_range(offsets, ans_start_char, ans_end_char)
    if ans_tok_end <= ans_tok_start or ctx_tok_end <= ctx_tok_start:
        return None

    out = model(input_ids=input_ids, output_attentions=True, use_cache=False)
    atts = out.attentions                                
    n_layers, n_heads = len(atts), atts[0].shape[1]
    n_ans = ans_tok_end - ans_tok_start


    ans_rows = torch.stack([a[0, :, ans_tok_start:ans_tok_end, :] for a in atts], dim=0)
    ctx_mean = ans_rows[..., ctx_tok_start:ctx_tok_end].mean(dim=-1)             
    ans_block = ans_rows[..., ans_tok_start:ans_tok_end]                        
    mask = torch.tril(torch.ones(n_ans, n_ans, device=device, dtype=ans_block.dtype), diagonal=-1)
    new_sum = (ans_block * mask).sum(dim=-1)                                     
    new_count = mask.sum(dim=-1).clamp_min(1.0)                                  
    new_mean = new_sum / new_count

    denom = ctx_mean + new_mean
    lookback = torch.where(denom > 0, ctx_mean / denom.clamp_min(1e-9), torch.zeros_like(ctx_mean))
    lookback[..., 0] = 1.0                               
    feats = lookback.permute(2, 0, 1).contiguous().to(torch.float32).cpu().numpy() 

  
    ans_offsets = np.array(offsets[ans_tok_start:ans_tok_end], dtype=np.int32)
    ans_offsets[:, 0] -= ans_start_char
    ans_offsets[:, 1] -= ans_start_char
    ans_offsets[:, 0] = np.clip(ans_offsets[:, 0], 0, len(output))
    ans_offsets[:, 1] = np.clip(ans_offsets[:, 1], 0, len(output))

    token_labels = _token_labels_from_spans(ans_offsets, output,
                                             record.get("hallucination_labels") or [])

    return RecordFeatures(
        record_id=record.get("id", ""),
        corruption_type=record["meta"]["corruption_type"],
        output=output,
        token_offsets=ans_offsets,
        features=feats,
        token_labels=token_labels,
        gold_spans=list(record.get("hallucination_labels") or []),
        truncated=truncated,
    )


def aggregate_window_features(rf, window_size):
    """Disjoint windows of `window_size` answer tokens.

    Returns
    -------
    X : (n_windows, L*H) float32
    y : (n_windows,) uint8 — 1 iff any token in window has label 1
    spans : (n_windows, 2) int32 — char offsets [start, end) of each window in rf.output
    """
    n_tok = rf.features.shape[0]
    if n_tok == 0:
        return (np.empty((0, 0), np.float32), np.empty((0,), np.uint8), np.empty((0, 2), np.int32))
    ws = max(1, int(window_size))
    n_win = (n_tok + ws - 1) // ws
    n_layers, n_heads = rf.features.shape[1], rf.features.shape[2]
    feats_flat = rf.features.reshape(n_tok, n_layers * n_heads)
    X = np.zeros((n_win, n_layers * n_heads), dtype=np.float32)
    y = np.zeros(n_win, dtype=np.uint8)
    spans = np.zeros((n_win, 2), dtype=np.int32)
    for w in range(n_win):
        lo, hi = w * ws, min(n_tok, (w + 1) * ws)
        X[w] = feats_flat[lo:hi].mean(axis=0)
        y[w] = int(rf.token_labels[lo:hi].max() > 0)
        spans[w, 0] = int(rf.token_offsets[lo, 0])
        spans[w, 1] = int(rf.token_offsets[hi - 1, 1])
    return X, y, spans

## Classifier and metrics

In [ ]:
@dataclass
class TrainedClassifier:
    model: LogisticRegression
    window_size: int
    threshold: float
    mean: np.ndarray
    std: np.ndarray

    def predict_proba(self, X):
        Xn = (X - self.mean) / np.where(self.std > 0, self.std, 1.0)
        return self.model.predict_proba(Xn)[:, 1]


def stack_window_dataset(records, window_size):
    """Concatenate window features across records → (X, y, index)."""
    X_chunks, y_chunks, idx = [], [], []
    for ri, rf in enumerate(records):
        X, y, spans = aggregate_window_features(rf, window_size)
        X_chunks.append(X); y_chunks.append(y)
        for wi in range(len(y)):
            idx.append((ri, wi, int(spans[wi, 1])))
    if not X_chunks or sum(c.shape[0] for c in X_chunks) == 0:
        return np.empty((0, 0), np.float32), np.empty((0,), np.uint8), idx
    return np.concatenate(X_chunks, axis=0), np.concatenate(y_chunks, axis=0), idx


def train_classifier(train_records, window_size=8, C=CLF_C, threshold=0.5,
                     class_weight=CLASS_WEIGHT, max_iter=1000, random_state=0):
    X, y, _ = stack_window_dataset(train_records, window_size)
    if X.size == 0:
        raise ValueError("no training windows extracted")
    if len(np.unique(y)) < 2:
        raise ValueError(f"training labels have a single class ({np.unique(y).tolist()})")
    mean, std = X.mean(axis=0), X.std(axis=0)
    Xn = (X - mean) / np.where(std > 0, std, 1.0)
    lr = LogisticRegression(C=C, max_iter=max_iter, class_weight=class_weight,
                            random_state=random_state, solver="lbfgs")
    lr.fit(Xn, y)
    return TrainedClassifier(model=lr, window_size=window_size, threshold=threshold,
                              mean=mean, std=std)


def predict_spans_for_record(rf, clf):
    """Per-window threshold + contiguous-run collapse → list of {start, end, text, confidence}."""
    X, _, spans = aggregate_window_features(rf, clf.window_size)
    if X.size == 0:
        return []
    probs = clf.predict_proba(X)
    pred_mask = probs >= clf.threshold
    out_spans, i = [], 0
    while i < len(pred_mask):
        if pred_mask[i]:
            j = i
            while j < len(pred_mask) and pred_mask[j]:
                j += 1
            start, end = int(spans[i, 0]), int(spans[j - 1, 1])
            if end > start:
                out_spans.append({
                    "start": start, "end": end,
                    "text": rf.output[start:end],
                    "confidence": float(probs[i:j].max()),
                })
            i = j
        else:
            i += 1
    return out_spans


def char_mask(text, spans):
    mask = [False] * len(text)
    for s in spans:
        a = max(0, int(s.get("start", 0)))
        b = min(len(text), int(s.get("end", 0)))
        for i in range(a, b):
            mask[i] = True
    return mask


def _prf1(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return {"precision": p, "recall": r, "f1": f, "tp": tp, "fp": fp, "fn": fn}


def example_level(per_record):
    """per_record: list of (gold_positive, pred_positive)."""
    tp = sum(1 for g, p in per_record if g and p)
    fp = sum(1 for g, p in per_record if (not g) and p)
    fn = sum(1 for g, p in per_record if g and (not p))
    out = _prf1(tp, fp, fn)
    out["n"] = len(per_record)
    out["n_positive"] = sum(1 for g, _ in per_record if g)
    return out


def span_level(per_record_masks):
    """per_record_masks: list of (gold_mask, pred_mask) over answer characters; micro-average."""
    tp = fp = fn = 0
    for gold, pred in per_record_masks:
        for g, p in zip(gold, pred):
            if p and g: tp += 1
            elif p and not g: fp += 1
            elif g and not p: fn += 1
    out = _prf1(tp, fp, fn)
    out["n"] = len(per_record_masks)
    return out


def score_results(per_record):
    """per_record: list of {output, gold_spans, pred_spans, corruption_type}."""
    by_type_ex = defaultdict(list)
    by_type_sp = defaultdict(list)
    overall_ex, overall_sp = [], []
    for r in per_record:
        gold_pos, pred_pos = bool(r["gold_spans"]), bool(r["pred_spans"])
        gold_mask = char_mask(r["output"], r["gold_spans"])
        pred_mask = char_mask(r["output"], r["pred_spans"])
        overall_ex.append((gold_pos, pred_pos))
        overall_sp.append((gold_mask, pred_mask))
        by_type_ex[r["corruption_type"]].append((gold_pos, pred_pos))
        by_type_sp[r["corruption_type"]].append((gold_mask, pred_mask))
    return {
        "example_level": {"overall": example_level(overall_ex),
                          "by_type": {t: example_level(v) for t, v in by_type_ex.items()}},
        "span_level": {"overall": span_level(overall_sp),
                       "by_type": {t: span_level(v) for t, v in by_type_sp.items()}},
    }

## RAGTruth adapter

In [4]:
RAGTRUTH_REPO = "wandb/RAGTruth-processed"


def _parse_labels(raw):
    """Parse RAGTruth's hallucination_labels (JSON-encoded list of dicts) into our schema."""
    if not raw:
        return []
    if isinstance(raw, str):
        try: labels = json.loads(raw)
        except json.JSONDecodeError: return []
    elif hasattr(raw, "tolist"):
        labels = raw.tolist()
    else:
        labels = list(raw)
    if not isinstance(labels, list):
        return []
    out = []
    for x in labels:
        if not isinstance(x, dict) or "start" not in x or "end" not in x:
            continue
        out.append({
            "start": int(x["start"]),
            "end": int(x["end"]),
            "text": str(x.get("text", "")),
            "label": str(x.get("label_type", x.get("label", "ragtruth"))),
        })
    return out


def load_ragtruth_records(split="train", *, max_records=None, quality="good", seed=0):
    """Load RAGTruth-processed into toolace-format records.

    `max_records` does stratified sampling preserving the natural positive rate.
    `quality` filters to `good` records (excludes 148 refused/truncated).
    """
    quality_set = {quality} if isinstance(quality, str) else set(quality)
    ds = load_dataset(RAGTRUTH_REPO, split=split)
    records = []
    for r in ds:
        if r["quality"] not in quality_set:
            continue
        labels = _parse_labels(r["hallucination_labels"])
        records.append({
            "id": f"ragtruth_{split}_{r['id']}",
            "query": r["query"],
            "context": r["context"],
            "output": r["output"],
            "hallucination_labels": labels,
            "meta": {
                "corruption_type": f"ragtruth_{r['task_type'].lower()}",
                "source": RAGTRUTH_REPO,
                "model": r.get("model"),
                "quality": r["quality"],
            },
        })
    if max_records is not None and max_records < len(records):
        pos = [r for r in records if r["hallucination_labels"]]
        neg = [r for r in records if not r["hallucination_labels"]]
        ratio = len(pos) / max(1, len(pos) + len(neg))
        n_pos = round(max_records * ratio)
        n_neg = max_records - n_pos
        rng = random.Random(seed)
        rng.shuffle(pos); rng.shuffle(neg)
        records = pos[:n_pos] + neg[:n_neg]
        rng.shuffle(records)
    return records

## Dataset loading

In [5]:
def list_split_parquets(repo_id, config, split):
    files = HfApi().list_repo_files(repo_id, repo_type="dataset")
    needles = (f"{config}/{split}-", f"{config}/{split}.parquet")
    return [
        f"hf://datasets/{repo_id}/{f}"
        for f in files
        if f.endswith(".parquet") and any(n in f for n in needles)
    ]


def load_config(repo_id, config, split):
    try:
        ds = load_dataset(repo_id, config, split=split)
        return [dict(row) for row in ds]
    except (ValueError, TypeError) as e:
        print(f"  load_dataset failed for {config}/{split}: {e}; falling back to parquet")
        uris = list_split_parquets(repo_id, config, split)
        if not uris:
            raise FileNotFoundError(f"no parquet for {repo_id}:{config}/{split}") from e
        df = pd.concat([pd.read_parquet(u) for u in uris], ignore_index=True)
        records = df.to_dict(orient="records")
        for r in records:
            for k in ("hallucination_labels", "meta"):
                if isinstance(r.get(k), str):
                    r[k] = json.loads(r[k])
                elif hasattr(r.get(k), "tolist"):
                    r[k] = r[k].tolist()
        return records


datasets: dict[tuple[str, str], list[dict]] = {}

print(f"loading {TRAIN_SOURCE_CONFIG}/train ...")
recs = load_config(DATASET_REPO, TRAIN_SOURCE_CONFIG, "train")
if MAX_TRAIN_RECORDS: recs = recs[:MAX_TRAIN_RECORDS]
datasets[(TRAIN_SOURCE_CONFIG, "train")] = recs

for cfg in CONFIGS:
    print(f"loading {cfg}/test ...")
    recs = load_config(DATASET_REPO, cfg, "test")
    if MAX_TEST_RECORDS: recs = recs[:MAX_TEST_RECORDS]
    datasets[(cfg, "test")] = recs

rows = []
for (cfg, split), recs in datasets.items():
    types = Counter(r["meta"]["corruption_type"] for r in recs)
    rows.append({"config": cfg, "split": split, "n": len(recs),
                 "clean": types.get("clean", 0),
                 "contradiction": types.get("contradiction", 0),
                 "missing_tool": types.get("missing_tool", 0),
                 "overgeneration": types.get("overgeneration", 0)})
pd.DataFrame(rows).set_index(["config", "split"])

loading combined/train ...
  load_dataset failed for combined/train: Feature type 'Json' not found. Available feature types: ['Value', 'ClassLabel', 'Translation', 'TranslationVariableLanguages', 'LargeList', 'List', 'Array2D', 'Array3D', 'Array4D', 'Array5D', 'Audio', 'Image', 'Video', 'Pdf']; falling back to parquet
loading combined/test ...
  load_dataset failed for combined/test: Feature type 'Json' not found. Available feature types: ['Value', 'ClassLabel', 'Translation', 'TranslationVariableLanguages', 'LargeList', 'List', 'Array2D', 'Array3D', 'Array4D', 'Array5D', 'Audio', 'Image', 'Video', 'Pdf']; falling back to parquet
loading contradiction/test ...
  load_dataset failed for contradiction/test: Feature type 'Json' not found. Available feature types: ['Value', 'ClassLabel', 'Translation', 'TranslationVariableLanguages', 'LargeList', 'List', 'Array2D', 'Array3D', 'Array4D', 'Array5D', 'Audio', 'Image', 'Video', 'Pdf']; falling back to parquet
loading missing_tool/test ...
  lo

n  clean  contradiction  missing_tool  overgeneration
config         split                                                          
combined       train  1955    267            292           562             834
               test    235     31             36            70              98
contradiction  test     67     31             36             0               0
missing_tool   test    101     31              0            70               0
overgeneration test    129     31              0             0              98

## LLM backbone loading

`Llama-2-7b-chat` 32 layers × 32 attention heads = 1024-dim feature vectors per answer-token window

In [6]:
t0 = time.time()
model, tokenizer = load_backbone(MODEL_NAME, device=DEVICE, dtype=torch.bfloat16)
print(f"loaded {MODEL_NAME} in {time.time() - t0:.1f}s")
print(f"  n_layers={model.config.num_hidden_layers}  n_heads={model.config.num_attention_heads}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

loaded NousResearch/Llama-2-7b-chat-hf in 8.0s
  n_layers=32  n_heads=32


## Extracting attention features

One teacher-forced forward pass per record

In [7]:
def cache_path(config, split):
    return FEATURE_CACHE / f"{config}__{split}.npz"


def save_features(path, feats):
    payload = {"n": np.array([len(feats)], dtype=np.int64)}
    for i, rf in enumerate(feats):
        payload[f"id_{i}"] = np.array(rf.record_id)
        payload[f"corr_{i}"] = np.array(rf.corruption_type)
        payload[f"out_{i}"] = np.array(rf.output)
        payload[f"offs_{i}"] = rf.token_offsets
        payload[f"feat_{i}"] = rf.features
        payload[f"lbl_{i}"] = rf.token_labels
        payload[f"gold_{i}"] = np.array(json.dumps(rf.gold_spans))
        payload[f"trunc_{i}"] = np.array([rf.truncated], dtype=np.bool_)
    np.savez_compressed(path, **payload)


def load_features(path):
    d = np.load(path, allow_pickle=False)
    n = int(d["n"][0])
    out = []
    for i in range(n):
        out.append(RecordFeatures(
            record_id=str(d[f"id_{i}"]),
            corruption_type=str(d[f"corr_{i}"]),
            output=str(d[f"out_{i}"]),
            token_offsets=d[f"offs_{i}"],
            features=d[f"feat_{i}"],
            token_labels=d[f"lbl_{i}"],
            gold_spans=json.loads(str(d[f"gold_{i}"])),
            truncated=bool(d[f"trunc_{i}"][0]),
        ))
    return out


def extract_split(config, split, records):
    path = cache_path(config, split)
    if path.exists():
        feats = load_features(path)
        print(f"  cached: {path.name}  n={len(feats)}")
        return feats
    feats, n_skipped, n_trunc = [], 0, 0
    for rec in tqdm(records, desc=f"{config}/{split}"):
        rf = extract_features(rec, model, tokenizer, max_length=MAX_LEN, device=DEVICE)
        if rf is None:
            n_skipped += 1
            continue
        n_trunc += int(rf.truncated)
        feats.append(rf)
    save_features(path, feats)
    print(f"  wrote {path.name}  n={len(feats)}  truncated={n_trunc}  skipped={n_skipped}")
    return feats


features_by_split: dict[tuple[str, str], list] = {}
for (cfg, split), recs in datasets.items():
    features_by_split[(cfg, split)] = extract_split(cfg, split, recs)

  cached: combined__train.npz  n=1955
  cached: combined__test.npz  n=235
  cached: contradiction__test.npz  n=67
  cached: missing_tool__test.npz  n=101
  cached: overgeneration__test.npz  n=129


In [8]:
RAGTRUTH_CACHE = FEATURE_CACHE / f"ragtruth_train__n{RAGTRUTH_TRAIN_RECORDS}.npz"


def extract_ragtruth_features():
    if RAGTRUTH_CACHE.exists():
        feats = load_features(RAGTRUTH_CACHE)
        print(f"cached: {RAGTRUTH_CACHE.name}  n={len(feats)}")
        return feats
    print(f"loading {RAGTRUTH_TRAIN_RECORDS} RAGTruth train records ...")
    records = load_ragtruth_records("train", max_records=RAGTRUTH_TRAIN_RECORDS, seed=0)
    print(f"  {len(records)} records "
          f"(pos rate: {sum(1 for r in records if r['hallucination_labels'])/len(records):.2f})")
    feats, n_skipped, n_trunc = [], 0, 0
    for rec in tqdm(records, desc="ragtruth/train"):
        rf = extract_features(rec, model, tokenizer, max_length=MAX_LEN, device=DEVICE)
        if rf is None:
            n_skipped += 1; continue
        n_trunc += int(rf.truncated)
        feats.append(rf)
    save_features(RAGTRUTH_CACHE, feats)
    print(f"wrote {RAGTRUTH_CACHE.name}  n={len(feats)}  truncated={n_trunc}  skipped={n_skipped}")
    return feats


ragtruth_train_feats = extract_ragtruth_features()
n_trunc = sum(1 for r in ragtruth_train_feats if r.truncated)
print(f"ragtruth train: {len(ragtruth_train_feats)} records  ({n_trunc} truncated to MAX_LEN={MAX_LEN})")

cached: ragtruth_train__n5000.npz  n=5000
ragtruth train: 5000 records  (0 truncated to MAX_LEN=4096)


## Classifier training

Four base classifiers: `{ragtruth, toolace} × {w=4, w=8}`. Same architecture for all (standardised features, balanced logistic regression)

In [9]:
toolace_train_feats = features_by_split[(TRAIN_SOURCE_CONFIG, "train")]
train_sources = {"ragtruth": ragtruth_train_feats, "toolace": toolace_train_feats}


def fit_and_report(name, train_feats, window_size):
    Xtr, ytr, _ = stack_window_dataset(train_feats, window_size)
    print(f"  [{name}] train windows: {Xtr.shape[0]}  feature_dim={Xtr.shape[1]}  "
          f"positive rate: {ytr.mean():.3f} ({int(ytr.sum())}/{len(ytr)})")
    t0 = time.time()
    clf = train_classifier(train_feats, window_size=window_size, C=CLF_C,
                            threshold=0.5, class_weight=CLASS_WEIGHT)
    print(f"  [{name}] fit logistic regression in {time.time() - t0:.1f}s")
    return clf


classifiers: dict[str, TrainedClassifier] = {}
for ws in WINDOW_SIZES:
    for src, feats in train_sources.items():
        key = f"lookbacklens-{src}-w{ws}"
        print(f"\n--- {key} ---")
        classifiers[key] = fit_and_report(key, feats, ws)
print(f"\ntrained {len(classifiers)} classifiers: {list(classifiers.keys())}")


--- lookbacklens-ragtruth-w4 ---
  [lookbacklens-ragtruth-w4] train windows: 247637  feature_dim=1024  positive rate: 0.069 (17134/247637)
  [lookbacklens-ragtruth-w4] fit logistic regression in 85.8s

--- lookbacklens-toolace-w4 ---
  [lookbacklens-toolace-w4] train windows: 75945  feature_dim=1024  positive rate: 0.140 (10633/75945)
  [lookbacklens-toolace-w4] fit logistic regression in 49.2s

--- lookbacklens-ragtruth-w8 ---
  [lookbacklens-ragtruth-w8] train windows: 125072  feature_dim=1024  positive rate: 0.083 (10408/125072)
  [lookbacklens-ragtruth-w8] fit logistic regression in 70.9s

--- lookbacklens-toolace-w8 ---
  [lookbacklens-toolace-w8] train windows: 38466  feature_dim=1024  positive rate: 0.159 (6135/38466)
  [lookbacklens-toolace-w8] fit logistic regression in 22.3s

trained 4 classifiers: ['lookbacklens-ragtruth-w4', 'lookbacklens-toolace-w4', 'lookbacklens-ragtruth-w8', 'lookbacklens-toolace-w8']


## Evaluation

For each of the 4 base classifiers and each threshold ∈ `THRESHOLDS`, predict per-window, collapse positives into char spans, score

In [10]:
all_metrics: dict[str, dict[str, dict]] = {}

for base_name, clf in classifiers.items():
    for t in THRESHOLDS:
        name = f"{base_name}-t{t:0.1f}"
        clf.threshold = t
        print(f"\n==== {name} ====")
        raw_dir = RAW_DIR / name
        raw_dir.mkdir(parents=True, exist_ok=True)
        method_metrics: dict[str, dict] = {}
        for cfg in CONFIGS:
            feats = features_by_split[(cfg, "test")]
            per_record = []
            for rf in feats:
                pred_spans = predict_spans_for_record(rf, clf)
                per_record.append({
                    "id": rf.record_id,
                    "corruption_type": rf.corruption_type,
                    "output": rf.output,
                    "gold_spans": rf.gold_spans,
                    "pred_spans": pred_spans,
                })
            with (raw_dir / f"{cfg}_test.jsonl").open("w") as f:
                for r in per_record:
                    f.write(json.dumps(r) + "\n")
            m = score_results(per_record)
            method_metrics[cfg] = m
            ex, sp = m["example_level"]["overall"], m["span_level"]["overall"]
            print(f"  {cfg:14s}  example F1={ex['f1']:.3f}  |  "
                  f"span P={sp['precision']:.3f} R={sp['recall']:.3f} F1={sp['f1']:.3f}  (n={ex['n']})")
        all_metrics[name] = method_metrics
        (OUTPUT_DIR / f"metrics_{name}.json").write_text(json.dumps(method_metrics, indent=2))

(OUTPUT_DIR / "metrics_all.json").write_text(json.dumps(all_metrics, indent=2))
print(f"\nproduced {len(all_metrics)} method variants")


==== lookbacklens-ragtruth-w4-t0.3 ====
  combined        example F1=0.929  |  span P=0.129 R=0.931 F1=0.227  (n=235)
  contradiction   example F1=0.699  |  span P=0.020 R=0.980 F1=0.040  (n=67)
  missing_tool    example F1=0.819  |  span P=0.073 R=0.974 F1=0.136  (n=101)
  overgeneration  example F1=0.863  |  span P=0.169 R=0.917 F1=0.285  (n=129)

==== lookbacklens-ragtruth-w4-t0.5 ====
  combined        example F1=0.929  |  span P=0.142 R=0.859 F1=0.244  (n=235)
  contradiction   example F1=0.699  |  span P=0.027 R=0.945 F1=0.052  (n=67)
  missing_tool    example F1=0.819  |  span P=0.081 R=0.926 F1=0.150  (n=101)
  overgeneration  example F1=0.863  |  span P=0.182 R=0.837 F1=0.299  (n=129)

==== lookbacklens-ragtruth-w4-t0.7 ====
  combined        example F1=0.929  |  span P=0.160 R=0.718 F1=0.262  (n=235)
  contradiction   example F1=0.706  |  span P=0.038 R=0.899 F1=0.073  (n=67)
  missing_tool    example F1=0.824  |  span P=0.092 R=0.800 F1=0.166  (n=101)
  overgeneration  exam

## Lexical floor (sanity)

A trivial baseline as sanity check with another baseline

In [11]:
WORD_RE = re.compile(r"[A-Za-z0-9]+")

def words_set(text):
    return {m.group(0).lower() for m in WORD_RE.finditer(text) if len(m.group(0)) > 2}

def lexical_predict_spans(context, output):
    ctx = words_set(context)
    mask = [False] * len(output)
    for m in WORD_RE.finditer(output):
        tok = m.group(0).lower()
        if len(tok) <= 2: continue
        if tok not in ctx:
            for i in range(m.start(), m.end()):
                mask[i] = True
    spans, i = [], 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j < len(mask) and mask[j]: j += 1
            spans.append({"start": i, "end": j, "text": output[i:j]})
            i = j
        else:
            i += 1
    return spans


lex_metrics: dict[str, dict] = {}
for cfg in CONFIGS:
    records = datasets[(cfg, "test")]
    per_record = []
    for r in records:
        per_record.append({
            "id": r.get("id"),
            "corruption_type": r["meta"]["corruption_type"],
            "output": r["output"],
            "gold_spans": r["hallucination_labels"],
            "pred_spans": lexical_predict_spans(r["context"], r["output"]),
        })
    lex_metrics[cfg] = score_results(per_record)
    ex, sp = lex_metrics[cfg]["example_level"]["overall"], lex_metrics[cfg]["span_level"]["overall"]
    print(f"  {cfg:14s}  example F1={ex['f1']:.3f}  span F1={sp['f1']:.3f}")

all_metrics["lexical"] = lex_metrics
(OUTPUT_DIR / "metrics_lexical.json").write_text(json.dumps(lex_metrics, indent=2))
(OUTPUT_DIR / "metrics_all.json").write_text(json.dumps(all_metrics, indent=2))

  combined        example F1=0.932  span F1=0.238
  contradiction   example F1=0.706  span F1=0.048
  missing_tool    example F1=0.824  span F1=0.140
  overgeneration  example F1=0.867  span F1=0.295


94761

## Results

In [12]:
def build_table(metrics, granularity):
    rows = []
    for method, by_cfg in metrics.items():
        row = {"method": method}
        for cfg in CONFIGS:
            m = by_cfg[cfg][granularity]["overall"]
            row[(cfg, "P")] = round(m["precision"], 4)
            row[(cfg, "R")] = round(m["recall"], 4)
            row[(cfg, "F1")] = round(m["f1"], 4)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("method")
    df.columns = pd.MultiIndex.from_tuples(df.columns)
    return df


example_table = build_table(all_metrics, "example_level")
span_table = build_table(all_metrics, "span_level")

print("Example-level (Table 2 analogue):")
display(example_table)
print("\nSpan-level char-overlap (Table 3 analogue):")
display(span_table)

example_table.to_csv(OUTPUT_DIR / "example_level.csv")
span_table.to_csv(OUTPUT_DIR / "span_level.csv")

Example-level (Table 2 analogue):


combined                 contradiction          \
                                     P       R      F1             P       R   
method                                                                         
lookbacklens-ragtruth-w4-t0.3   0.8681  1.0000  0.9294        0.5373  1.0000   
lookbacklens-ragtruth-w4-t0.5   0.8681  1.0000  0.9294        0.5373  1.0000   
lookbacklens-ragtruth-w4-t0.7   0.8712  0.9951  0.9291        0.5455  1.0000   
lookbacklens-toolace-w4-t0.3    0.8755  1.0000  0.9336        0.5538  1.0000   
lookbacklens-toolace-w4-t0.5    0.8831  1.0000  0.9379        0.5714  1.0000   
lookbacklens-toolace-w4-t0.7    0.9054  0.9853  0.9437        0.6250  0.9722   
lookbacklens-ragtruth-w8-t0.3   0.8681  1.0000  0.9294        0.5373  1.0000   
lookbacklens-ragtruth-w8-t0.5   0.8681  1.0000  0.9294        0.5373  1.0000   
lookbacklens-ragtruth-w8-t0.7   0.8783  0.9902  0.9309        0.5625  1.0000   
lookbacklens-toolace-w8-t0.3    0.8793  1.0000  0.9358        0.5625  1.0000   
lookbacklens-toolace-w8-t0.5    0.8938  0.9902  0.9395        0.5932  0.9722   
lookbacklens-toolace-w8-t0.7    0.9078  0.9657  0.9359        0.6226  0.9167   
lexical                         0.8718  1.0000  0.9315        0.5455  1.0000   

                                      missing_tool                  \
                                   F1            P       R      F1   
method                                                               
lookbacklens-ragtruth-w4-t0.3  0.6990       0.6931  1.0000  0.8187   
lookbacklens-ragtruth-w4-t0.5  0.6990       0.6931  1.0000  0.8187   
lookbacklens-ragtruth-w4-t0.7  0.7059       0.7000  1.0000  0.8235   
lookbacklens-toolace-w4-t0.3   0.7129       0.7071  1.0000  0.8284   
lookbacklens-toolace-w4-t0.5   0.7273       0.7216  1.0000  0.8383   
lookbacklens-toolace-w4-t0.7   0.7609       0.7692  1.0000  0.8696   
lookbacklens-ragtruth-w8-t0.3  0.6990       0.6931  1.0000  0.8187   
lookbacklens-ragtruth-w8-t0.5  0.6990       0.6931  1.0000  0.8187   
lookbacklens-ragtruth-w8-t0.7  0.7200       0.7083  0.9714  0.8193   
lookbacklens-toolace-w8-t0.3   0.7200       0.7143  1.0000  0.8333   
lookbacklens-toolace-w8-t0.5   0.7368       0.7447  1.0000  0.8537   
lookbacklens-toolace-w8-t0.7   0.7416       0.7778  1.0000  0.8750   
lexical                        0.7059       0.7000  1.0000  0.8235   

                              overgeneration                  
                                           P       R      F1  
method                                                        
lookbacklens-ragtruth-w4-t0.3         0.7597  1.0000  0.8634  
lookbacklens-ragtruth-w4-t0.5         0.7597  1.0000  0.8634  
lookbacklens-ragtruth-w4-t0.7         0.7638  0.9898  0.8622  
lookbacklens-toolace-w4-t0.3          0.7717  1.0000  0.8711  
lookbacklens-toolace-w4-t0.5          0.7840  1.0000  0.8789  
lookbacklens-toolace-w4-t0.7          0.8205  0.9796  0.8930  
lookbacklens-ragtruth-w8-t0.3         0.7597  1.0000  0.8634  
lookbacklens-ragtruth-w8-t0.5         0.7597  1.0000  0.8634  
lookbacklens-ragtruth-w8-t0.7         0.7778  1.0000  0.8750  
lookbacklens-toolace-w8-t0.3          0.7778  1.0000  0.8750  
lookbacklens-toolace-w8-t0.5          0.8017  0.9898  0.8858  
lookbacklens-toolace-w8-t0.7          0.8246  0.9592  0.8868  
lexical                               0.7656  1.0000  0.8673


Span-level char-overlap (Table 3 analogue):


combined                 contradiction          \
                                     P       R      F1             P       R   
method                                                                         
lookbacklens-ragtruth-w4-t0.3   0.1291  0.9306  0.2268        0.0204  0.9799   
lookbacklens-ragtruth-w4-t0.5   0.1425  0.8589  0.2444        0.0270  0.9447   
lookbacklens-ragtruth-w4-t0.7   0.1600  0.7184  0.2618        0.0380  0.8995   
lookbacklens-toolace-w4-t0.3    0.2452  0.9148  0.3867        0.0323  0.9045   
lookbacklens-toolace-w4-t0.5    0.3360  0.8356  0.4793        0.0506  0.8719   
lookbacklens-toolace-w4-t0.7    0.4571  0.7028  0.5539        0.0815  0.7261   
lookbacklens-ragtruth-w8-t0.3   0.1302  0.9252  0.2283        0.0206  0.9698   
lookbacklens-ragtruth-w8-t0.5   0.1464  0.8644  0.2504        0.0263  0.9271   
lookbacklens-ragtruth-w8-t0.7   0.1651  0.7382  0.2698        0.0339  0.8291   
lookbacklens-toolace-w8-t0.3    0.2307  0.9265  0.3694        0.0313  0.9724   
lookbacklens-toolace-w8-t0.5    0.3019  0.8183  0.4411        0.0430  0.8744   
lookbacklens-toolace-w8-t0.7    0.3926  0.6921  0.5010        0.0644  0.7462   
lexical                         0.1437  0.6975  0.2383        0.0249  0.7739   

                                      missing_tool                  \
                                   F1            P       R      F1   
method                                                               
lookbacklens-ragtruth-w4-t0.3  0.0400       0.0732  0.9736  0.1362   
lookbacklens-ragtruth-w4-t0.5  0.0524       0.0814  0.9264  0.1496   
lookbacklens-ragtruth-w4-t0.7  0.0730       0.0925  0.8001  0.1658   
lookbacklens-toolace-w4-t0.3   0.0624       0.1524  0.9994  0.2645   
lookbacklens-toolace-w4-t0.5   0.0956       0.2253  0.9988  0.3676   
lookbacklens-toolace-w4-t0.7   0.1465       0.3594  0.9933  0.5278   
lookbacklens-ragtruth-w8-t0.3  0.0404       0.0730  0.9603  0.1357   
lookbacklens-ragtruth-w8-t0.5  0.0511       0.0816  0.9073  0.1498   
lookbacklens-ragtruth-w8-t0.7  0.0651       0.0914  0.7798  0.1637   
lookbacklens-toolace-w8-t0.3   0.0607       0.1437  0.9997  0.2513   
lookbacklens-toolace-w8-t0.5   0.0820       0.2087  0.9986  0.3452   
lookbacklens-toolace-w8-t0.7   0.1186       0.3070  0.9693  0.4663   
lexical                        0.0483       0.0778  0.7086  0.1401   

                              overgeneration                  
                                           P       R      F1  
method                                                        
lookbacklens-ragtruth-w4-t0.3         0.1685  0.9170  0.2848  
lookbacklens-ragtruth-w4-t0.5         0.1818  0.8372  0.2987  
lookbacklens-ragtruth-w4-t0.7         0.2003  0.6896  0.3104  
lookbacklens-toolace-w4-t0.3          0.3154  0.8914  0.4660  
lookbacklens-toolace-w4-t0.5          0.4219  0.7887  0.5497  
lookbacklens-toolace-w4-t0.7          0.5444  0.6206  0.5800  
lookbacklens-ragtruth-w8-t0.3         0.1702  0.9139  0.2870  
lookbacklens-ragtruth-w8-t0.5         0.1884  0.8503  0.3084  
lookbacklens-ragtruth-w8-t0.7         0.2104  0.7236  0.3260  
lookbacklens-toolace-w8-t0.3          0.2996  0.9045  0.4501  
lookbacklens-toolace-w8-t0.5          0.3790  0.7659  0.5071  
lookbacklens-toolace-w8-t0.7          0.4669  0.6127  0.5299  
lexical                               0.1875  0.6919  0.2950

## Per-corruption-type breakdown

`clean` records have no gold positives so span-level F1 is 0/undefined there on clean records, example-level precision = 1 − FPR

In [13]:
def by_type_table(metrics, granularity, cfg="combined"):
    rows = []
    for method, by_cfg in metrics.items():
        for ctype, m in by_cfg[cfg][granularity]["by_type"].items():
            rows.append({"method": method, "corruption_type": ctype, "n": m["n"],
                         "precision": round(m["precision"], 4),
                         "recall": round(m["recall"], 4),
                         "f1": round(m["f1"], 4)})
    return pd.DataFrame(rows).pivot_table(
        index="method", columns="corruption_type",
        values=["precision", "recall", "f1"], aggfunc="first")


print("Per-type, example-level (combined config):")
display(by_type_table(all_metrics, "example_level", "combined"))
print("\nPer-type, span-level (combined config):")
display(by_type_table(all_metrics, "span_level", "combined"))

Per-type, example-level (combined config):


f1                                            \
corruption_type               clean contradiction missing_tool overgeneration   
method                                                                          
lexical                         0.0        1.0000       1.0000         1.0000   
lookbacklens-ragtruth-w4-t0.3   0.0        1.0000       1.0000         1.0000   
lookbacklens-ragtruth-w4-t0.5   0.0        1.0000       1.0000         1.0000   
lookbacklens-ragtruth-w4-t0.7   0.0        1.0000       1.0000         0.9949   
lookbacklens-ragtruth-w8-t0.3   0.0        1.0000       1.0000         1.0000   
lookbacklens-ragtruth-w8-t0.5   0.0        1.0000       1.0000         1.0000   
lookbacklens-ragtruth-w8-t0.7   0.0        1.0000       0.9855         1.0000   
lookbacklens-toolace-w4-t0.3    0.0        1.0000       1.0000         1.0000   
lookbacklens-toolace-w4-t0.5    0.0        1.0000       1.0000         1.0000   
lookbacklens-toolace-w4-t0.7    0.0        0.9859       1.0000         0.9897   
lookbacklens-toolace-w8-t0.3    0.0        1.0000       1.0000         1.0000   
lookbacklens-toolace-w8-t0.5    0.0        0.9859       1.0000         0.9949   
lookbacklens-toolace-w8-t0.7    0.0        0.9565       1.0000         0.9792   

                              precision                             \
corruption_type                   clean contradiction missing_tool   
method                                                               
lexical                             0.0           1.0          1.0   
lookbacklens-ragtruth-w4-t0.3       0.0           1.0          1.0   
lookbacklens-ragtruth-w4-t0.5       0.0           1.0          1.0   
lookbacklens-ragtruth-w4-t0.7       0.0           1.0          1.0   
lookbacklens-ragtruth-w8-t0.3       0.0           1.0          1.0   
lookbacklens-ragtruth-w8-t0.5       0.0           1.0          1.0   
lookbacklens-ragtruth-w8-t0.7       0.0           1.0          1.0   
lookbacklens-toolace-w4-t0.3        0.0           1.0          1.0   
lookbacklens-toolace-w4-t0.5        0.0           1.0          1.0   
lookbacklens-toolace-w4-t0.7        0.0           1.0          1.0   
lookbacklens-toolace-w8-t0.3        0.0           1.0          1.0   
lookbacklens-toolace-w8-t0.5        0.0           1.0          1.0   
lookbacklens-toolace-w8-t0.7        0.0           1.0          1.0   

                                             recall                \
corruption_type               overgeneration  clean contradiction   
method                                                              
lexical                                  1.0    0.0        1.0000   
lookbacklens-ragtruth-w4-t0.3            1.0    0.0        1.0000   
lookbacklens-ragtruth-w4-t0.5            1.0    0.0        1.0000   
lookbacklens-ragtruth-w4-t0.7            1.0    0.0        1.0000   
lookbacklens-ragtruth-w8-t0.3            1.0    0.0        1.0000   
lookbacklens-ragtruth-w8-t0.5            1.0    0.0        1.0000   
lookbacklens-ragtruth-w8-t0.7            1.0    0.0        1.0000   
lookbacklens-toolace-w4-t0.3             1.0    0.0        1.0000   
lookbacklens-toolace-w4-t0.5             1.0    0.0        1.0000   
lookbacklens-toolace-w4-t0.7             1.0    0.0        0.9722   
lookbacklens-toolace-w8-t0.3             1.0    0.0        1.0000   
lookbacklens-toolace-w8-t0.5             1.0    0.0        0.9722   
lookbacklens-toolace-w8-t0.7             1.0    0.0        0.9167   

                                                           
corruption_type               missing_tool overgeneration  
method                                                     
lexical                             1.0000         1.0000  
lookbacklens-ragtruth-w4-t0.3       1.0000         1.0000  
lookbacklens-ragtruth-w4-t0.5       1.0000         1.0000  
lookbacklens-ragtruth-w4-t0.7       1.0000         0.9898  
lookbacklens-ragtruth-w8-t0.3       1.0000         1.0000  
lookbacklens-ragtruth-w8-t0.5 


Per-type, span-level (combined config):


f1                                            \
corruption_type               clean contradiction missing_tool overgeneration   
method                                                                          
lexical                         0.0        0.0838       0.1658         0.3254   
lookbacklens-ragtruth-w4-t0.3   0.0        0.0664       0.1616         0.3156   
lookbacklens-ragtruth-w4-t0.5   0.0        0.0931       0.1754         0.3285   
lookbacklens-ragtruth-w4-t0.7   0.0        0.1365       0.1921         0.3387   
lookbacklens-ragtruth-w8-t0.3   0.0        0.0675       0.1611         0.3182   
lookbacklens-ragtruth-w8-t0.5   0.0        0.0906       0.1763         0.3399   
lookbacklens-ragtruth-w8-t0.7   0.0        0.1197       0.1905         0.3561   
lookbacklens-toolace-w4-t0.3    0.0        0.0972       0.3144         0.5108   
lookbacklens-toolace-w4-t0.5    0.0        0.1456       0.4241         0.5915   
lookbacklens-toolace-w4-t0.7    0.0        0.2059       0.5784         0.6062   
lookbacklens-toolace-w8-t0.3    0.0        0.0917       0.2981         0.4930   
lookbacklens-toolace-w8-t0.5    0.0        0.1233       0.4024         0.5490   
lookbacklens-toolace-w8-t0.7    0.0        0.1737       0.5242         0.5612   

                              precision                             \
corruption_type                   clean contradiction missing_tool   
method                                                               
lexical                             0.0        0.0443       0.0939   
lookbacklens-ragtruth-w4-t0.3       0.0        0.0344       0.0881   
lookbacklens-ragtruth-w4-t0.5       0.0        0.0489       0.0968   
lookbacklens-ragtruth-w4-t0.7       0.0        0.0739       0.1092   
lookbacklens-ragtruth-w8-t0.3       0.0        0.0350       0.0879   
lookbacklens-ragtruth-w8-t0.5       0.0        0.0476       0.0976   
lookbacklens-ragtruth-w8-t0.7       0.0        0.0645       0.1085   
lookbacklens-toolace-w4-t0.3        0.0        0.0513       0.1865   
lookbacklens-toolace-w4-t0.5        0.0        0.0794       0.2692   
lookbacklens-toolace-w4-t0.7        0.0        0.1200       0.4080   
lookbacklens-toolace-w8-t0.3        0.0        0.0481       0.1752   
lookbacklens-toolace-w8-t0.5        0.0        0.0663       0.2520   
lookbacklens-toolace-w8-t0.7        0.0        0.0983       0.3593   

                                             recall                \
corruption_type               overgeneration  clean contradiction   
method                                                              
lexical                               0.2128    0.0        0.7739   
lookbacklens-ragtruth-w4-t0.3         0.1906    0.0        0.9799   
lookbacklens-ragtruth-w4-t0.5         0.2043    0.0        0.9447   
lookbacklens-ragtruth-w4-t0.7         0.2245    0.0        0.8995   
lookbacklens-ragtruth-w8-t0.3         0.1926    0.0        0.9698   
lookbacklens-ragtruth-w8-t0.5         0.2124    0.0        0.9271   
lookbacklens-ragtruth-w8-t0.7         0.2362    0.0        0.8291   
lookbacklens-toolace-w4-t0.3          0.3580    0.0        0.9045   
lookbacklens-toolace-w4-t0.5          0.4731    0.0        0.8719   
lookbacklens-toolace-w4-t0.7          0.5925    0.0        0.7261   
lookbacklens-toolace-w8-t0.3          0.3389    0.0        0.9724   
lookbacklens-toolace-w8-t0.5          0.4279    0.0        0.8744   
lookbacklens-toolace-w8-t0.7          0.5177    0.0        0.7462   

                                                           
corruption_type               missing_tool overgeneration  
method                                                     
lexical                             0.7086         0.6919  
lookbacklens-ragtruth-w4-t0.3       0.9736         0.9170  
lookbacklens-ragtruth-w4-t0.5       0.9264         0.8372  
lookbacklens-ragtruth-w4-t0.7       0.8001         0.6896  
lookbacklens-ragtruth-w8-t0.3       0.9603         0.9139  
lookbacklens-ragtruth-w8-t0.5 

## Focused side-by-side comparison

This section reloads `metrics_all.json` from disk (so it works even if §10 is skipped) and presents the headline comparison in a human-readable way:

1. **Best-of-sweep table** — for each (training-source) pair, the (window, threshold) that maximises span F1 per config.
2. **LookbackLens vs LettuceDetect** — adds the LettuceDetect-base/large numbers from correspondng notebook so the apples-to-apples zero-shot comparison is in one place.
3. **Threshold sweep visualisation** — span P / R / F1 vs threshold for the strongest variant.
4. **Per-corruption-type detail** — which configs benefit most from which variant.

Numbers come from the `metrics_all.json` produced above; everything else is just rendering.

In [ ]:
metrics = json.loads((OUTPUT_DIR / "metrics_all.json").read_text())
print(f"loaded metrics for {len(metrics)} methods × {len(CONFIGS)} configs from {OUTPUT_DIR / 'metrics_all.json'}")

def parse_method(name):
    """`lookbacklens-toolace-w4-t0.7` -> dict, or `lexical` -> all-None."""
    if name == "lexical":
        return {"family": "lexical", "source": None, "window": None, "threshold": None}
    parts = name.split("-")
    return {
        "family": "lookbacklens",
        "source": parts[1],
        "window": int(parts[2][1:]),
        "threshold": float(parts[3][1:]),
    }


rows = []
for method, by_cfg in metrics.items():
    meta = parse_method(method)
    for cfg, m in by_cfg.items():
        ex, sp = m["example_level"]["overall"], m["span_level"]["overall"]
        rows.append({"method": method, **meta, "config": cfg,
                      "example_F1": ex["f1"],
                      "span_P": sp["precision"], "span_R": sp["recall"], "span_F1": sp["f1"]})
results_df = pd.DataFrame(rows)
print(f"\nlong-form dataframe: {results_df.shape[0]} rows × {results_df.shape[1]} cols")
results_df.head()

loaded metrics for 13 methods × 4 configs from /home/mvlsmirnov/skoltech/llm3/halu-toolace/notebooks/results/lookbacklens_baseline/metrics_all.json

long-form dataframe: 52 rows × 10 cols


,method,family,source,window,threshold,config,example_F1,span_P,span_R,span_F1
0,lookbacklens-ragtruth-w4-t0.3,lookbacklens,ragtruth,4.0,0.3,combined,0.929385,0.129113,0.930631,0.226766
1,lookbacklens-ragtruth-w4-t0.3,lookbacklens,ragtruth,4.0,0.3,contradiction,0.699029,0.020425,0.979899,0.040016
2,lookbacklens-ragtruth-w4-t0.3,lookbacklens,ragtruth,4.0,0.3,missing_tool,0.818713,0.073234,0.973638,0.136222
3,lookbacklens-ragtruth-w4-t0.3,lookbacklens,ragtruth,4.0,0.3,overgeneration,0.863436,0.168546,0.916978,0.284752
4,lookbacklens-ragtruth-w4-t0.5,lookbacklens,ragtruth,4.0,0.5,combined,0.929385,0.142492,0.858911,0.244433


### (1) Best-of-sweep table per training source

For each `(source ∈ {ragtruth, toolace})` family, pick the (window, threshold) that maximises span F1 *per config* — the operating point a deployer would choose for that target. (In our run it's always `t=0.7`, with `w=4` for in-domain and `w=8` for zero-shot.)

In [15]:
def best_per_source_config(df, granularity_col="span_F1"):
    lb = df[df["family"] == "lookbacklens"].copy()
    out_rows = []
    for source, group_src in lb.groupby("source"):
        for cfg, group in group_src.groupby("config"):
            best = group.loc[group[granularity_col].idxmax()]
            out_rows.append({
                "source": source, "config": cfg,
                "best_window": int(best["window"]),
                "best_threshold": best["threshold"],
                "span_P": round(best["span_P"], 3),
                "span_R": round(best["span_R"], 3),
                "span_F1": round(best["span_F1"], 3),
                "example_F1": round(best["example_F1"], 3),
            })
    return pd.DataFrame(out_rows)


best_df = best_per_source_config(results_df)
print("Best (window, threshold) per (source, config) — span F1:\n")
display(best_df)

Best (window, threshold) per (source, config) — span F1:



,source,config,best_window,best_threshold,span_P,span_R,span_F1,example_F1
0,ragtruth,combined,8,0.7,0.165,0.738,0.270,0.931
1,ragtruth,contradiction,4,0.7,0.038,0.899,0.073,0.706
2,ragtruth,missing_tool,4,0.7,0.092,0.800,0.166,0.824
3,ragtruth,overgeneration,8,0.7,0.210,0.724,0.326,0.875
4,toolace,combined,4,0.7,0.457,0.703,0.554,0.944
5,toolace,contradiction,4,0.7,0.081,0.726,0.147,0.761
6,toolace,missing_tool,4,0.7,0.359,0.993,0.528,0.870
7,toolace,overgeneration,4,0.7,0.544,0.621,0.580,0.893


### (2) Side-by-side with LettuceDetect (zero-shot vs zero-shot)

Hard-coded LettuceDetect-base/large numbers from `notebooks/lettucedetect_baseline.ipynb` (same toolace test split, same metrics). The apples-to-apples comparison row is **lookbacklens-ragtruth** (also zero-shot from RAGTruth)

In [ ]:
LETTUCE = {
    "lettucedetect-base":  {"combined": 0.282, "contradiction": 0.068, "missing_tool": 0.193, "overgeneration": 0.328},
    "lettucedetect-large": {"combined": 0.286, "contradiction": 0.080, "missing_tool": 0.209, "overgeneration": 0.322},
}

def best_F1_row(source):
    rows = []
    lb = results_df[(results_df["family"] == "lookbacklens") & (results_df["source"] == source)]
    for cfg in CONFIGS:
        sub = lb[lb["config"] == cfg]
        best = sub.loc[sub["span_F1"].idxmax()]
        rows.append({"config": cfg, "span_F1": round(best["span_F1"], 3),
                      "at": f"w={int(best['window'])}, t={best['threshold']:0.1f}"})
    return pd.DataFrame(rows).set_index("config")

lex_row = {cfg: round(results_df[(results_df["family"] == "lexical") & (results_df["config"] == cfg)]["span_F1"].iloc[0], 3) for cfg in CONFIGS}

comp = pd.DataFrame({
    "lexical floor":                     [lex_row[c] for c in CONFIGS],
    "lettucedetect-base (zero-shot)":    [LETTUCE["lettucedetect-base"][c] for c in CONFIGS],
    "lettucedetect-large (zero-shot)":   [LETTUCE["lettucedetect-large"][c] for c in CONFIGS],
    "lookbacklens-ragtruth (best, zero-shot)": [best_F1_row("ragtruth").loc[c, "span_F1"] for c in CONFIGS],
    "lookbacklens-toolace (best, in-domain)":  [best_F1_row("toolace").loc[c, "span_F1"] for c in CONFIGS],
}, index=CONFIGS).T
comp.columns.name = "config"

print("Span-level F1 — LookbackLens vs LettuceDetect (rows: method, cols: config)")
display(comp.round(3))

Span-level F1 — LookbackLens vs LettuceDetect (rows: method, cols: config)


config,combined,contradiction,missing_tool,overgeneration
lexical floor,0.238,0.048,0.140,0.295
lettucedetect-base (zero-shot),0.282,0.068,0.193,0.328
lettucedetect-large (zero-shot),0.286,0.080,0.209,0.322
"lookbacklens-ragtruth (best, zero-shot)",0.270,0.073,0.166,0.326
"lookbacklens-toolace (best, in-domain)",0.554,0.147,0.528,0.580


### (3) Threshold sweep visualisation (best in-domain variant)

For `lookbacklens-toolace-w4`, span P / R / F1 vs threshold on each config. Recovers the headline finding: at `t=0.7` precision climbs by +21 pp on `combined` while recall only drops by 13 pp → F1 wins.

In [ ]:
sweep = results_df[(results_df["family"] == "lookbacklens")
                    & (results_df["source"] == "toolace")
                    & (results_df["window"] == 4)].copy()
sweep = sweep.sort_values("threshold")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(CONFIGS), figsize=(16, 3.5), sharey=True)
for ax, cfg in zip(axes, CONFIGS):
    sub = sweep[sweep["config"] == cfg]
    ax.plot(sub["threshold"], sub["span_P"], "o-", label="precision")
    ax.plot(sub["threshold"], sub["span_R"], "s-", label="recall")
    ax.plot(sub["threshold"], sub["span_F1"], "^-", label="F1", linewidth=2)
    ax.set_title(f"toolace-w4 / {cfg}")
    ax.set_xlabel("threshold")
    ax.set_xticks([0.3, 0.5, 0.7])
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("score")
axes[0].legend(loc="lower left")
plt.tight_layout()
plt.show()

### (4) Per-corruption-type detail on `combined`

How each method does on each kind of injected hallucination (clean / contradiction / missing_tool / overgeneration), pulled from the `combined` config's `by_type` breakdown.

In [18]:
def by_type_long(metrics, cfg="combined"):
    rows = []
    for method, by_cfg in metrics.items():
        for ctype, m in by_cfg[cfg]["span_level"]["by_type"].items():
            rows.append({"method": method, "corruption_type": ctype,
                         "n": m["n"], "span_F1": round(m["f1"], 3)})
    return pd.DataFrame(rows)


by_type = by_type_long(metrics, "combined")
# Pivot: rows = method, cols = corruption_type, values = span F1
pivot = by_type.pivot(index="method", columns="corruption_type", values="span_F1")
# Sort by overall combined span F1 descending
order_idx = pd.DataFrame([{"method": m, "f1": metrics[m]["combined"]["span_level"]["overall"]["f1"]}
                           for m in pivot.index]).sort_values("f1", ascending=False)["method"].tolist()
pivot = pivot.loc[order_idx]

print("Per-corruption-type span F1 on `combined` (sorted by overall combined F1):")
display(pivot.round(3))

Per-corruption-type span F1 on `combined` (sorted by overall combined F1):


corruption_type,clean,contradiction,missing_tool,overgeneration
method,,,,
lookbacklens-toolace-w4-t0.7,0.0,0.206,0.578,0.606
lookbacklens-toolace-w8-t0.7,0.0,0.174,0.524,0.561
lookbacklens-toolace-w4-t0.5,0.0,0.146,0.424,0.591
lookbacklens-toolace-w8-t0.5,0.0,0.123,0.402,0.549
lookbacklens-toolace-w4-t0.3,0.0,0.097,0.314,0.511
lookbacklens-toolace-w8-t0.3,0.0,0.092,0.298,0.493
lookbacklens-ragtruth-w8-t0.7,0.0,0.120,0.190,0.356
lookbacklens-ragtruth-w4-t0.7,0.0,0.137,0.192,0.339
lookbacklens-ragtruth-w8-t0.5,0.0,0.091,0.176,0.340


### (5) Qualitative inspection (one record)

A single non-clean record from `combined/test` with gold spans (red, underlined) vs the best in-domain method's predicted spans (yellow). Green where TP overlap. Reloaded from disk so this works without rerunning inference.

In [ ]:
qual_method = f"lookbacklens-toolace-w{WINDOW_SIZES[0]}-t0.7"  
raw_path = RAW_DIR / qual_method / "combined_test.jsonl"

with raw_path.open() as f:
    sample_records = [json.loads(l) for l in f]


def highlight(output, gold, pred):
    g = char_mask(output, gold)
    p = char_mask(output, pred)
    parts = []
    for i, ch in enumerate(output):
        ch_h = html.escape(ch)
        if g[i] and p[i]:
            parts.append(f"<span style=\"background:#9f9\">{ch_h}</span>")
        elif g[i]:
            parts.append(f"<span style=\"background:#fdd;text-decoration:underline\">{ch_h}</span>")
        elif p[i]:
            parts.append(f"<span style=\"background:#ffd\">{ch_h}</span>")
        else:
            parts.append(ch_h)
    return "".join(parts)


from IPython.display import HTML
examples = [r for r in sample_records if r["corruption_type"] != "clean"][:5]
blocks = [
    f"<style>body{{font-family:system-ui}}</style>"
    f"<p><b>{qual_method}</b> &mdash; legend: "
    f"<span style='background:#9f9'>TP</span> "
    f"<span style='background:#fdd;text-decoration:underline'>FN (gold only)</span> "
    f"<span style='background:#ffd'>FP (pred only)</span></p>"
]
for r in examples:
    blocks.append(f"<hr><b>{r['id']}</b> &mdash; corruption_type=<code>{r['corruption_type']}</code>")
    blocks.append(f"<div style='white-space:pre-wrap'>{highlight(r['output'], r['gold_spans'], r['pred_spans'])}</div>")
display(HTML("\n".join(blocks)))

## 15. Discussion

The primary baseline (`lookbacklens-ragtruth-*`) is trained on the same RAGTruth corpus the published LettuceDetect checkpoints used, so the two methods are reported under the same **zero-shot domain-transfer** protocol

### Method recap

- **Backbone:** frozen `Llama-2-7b-chat` (paper's exact backbone), eager attention.
- **Feature:** per-answer-token, per-(layer × head) lookback ratio. 32 × 32 = 1024-dim per token (matches paper)
- **Aggregation:** disjoint **`w=4`** and **`w=8`** token windows reported side-by-side; window feature = mean of token features; window label = `any(token_labels)`.
- **Classifier:** logistic regression (`balanced` class weight, `C=1`); evaluated at three thresholds **`t ∈ {0.3, 0.5, 0.7}`**.
- **Two training-data variants × two window sizes × three thresholds = twelve method rows.**
- **Prediction:** predicted-positive windows collapsed to contiguous char spans; same RAGTruth-style char-overlap scorer as LettuceDetect.

### Measured results (best operating point per variant)

| Method | best (w, t) | combined | contradiction | missing_tool | overgeneration |
|---|---|---:|---:|---:|---:|
| lexical                                            | —        | 0.238 | 0.048 | 0.140 | 0.295 |
| lettucedect-base (zero-shot)            | —        | 0.282 | 0.068 | 0.193 | 0.328 |
| lettucedect-large (zero-shot)           | —        | 0.286 | 0.080 | 0.209 | 0.322 |
| lookbacklens-ragtruth (zero-shot, best at t=0.7)   | w=8, t=0.7 | 0.270 | 0.073 | 0.166 | 0.326 |
| **lookbacklens-toolace** (in-domain, best at t=0.7) | **w=4, t=0.7** | **0.554** | **0.147** | **0.528** | **0.580** |

### What these numbers say

1. **Threshold is the bigger lever than window size.** Going from `t=0.5 → t=0.7` lifts in-domain span F1 by +7-16 pp depending on config (combined +7.5, missing_tool **+16**, contradiction **+5**, overgeneration +3); going from `w=8 → w=4` lifts it by 1-4 pp. The two are complementary (best is `w=4, t=0.7` everywhere) but threshold sweeping is by far the bigger win.

2. **Threshold also helps zero-shot — but less.** `ragtruth-w8: t=0.5 → t=0.7` lifts span F1 by +2.0 pp on combined, +3.0 pp on missing_tool — useful but small. The ragtruth-trained classifier's probabilities are less well-calibrated for toolace, so threshold has less leverage.

3. **Recall stays high even at t=0.7.** On `missing_tool` the in-domain classifier still gets R=0.99 at t=0.7 (vs R=1.00 at t=0.5) — the probability mass on positive windows is heavily concentrated above 0.7. The classifier *knows* what it's doing; the default 0.5 was just emitting borderline-probable spans that didn't pull their weight.

4. **`lookbacklens-toolace-w4-t0.7` is the strongest LookbackLens result by far.** Span F1 0.554 on combined beats LettuceDetect-large (0.286) by **+27 pp**, on overgeneration 0.580 vs 0.322 (**+26 pp**), on missing_tool 0.528 vs 0.209 (**+32 pp**), on contradiction 0.147 vs 0.080 (**+7 pp**). Every type is now a clear win.

5. **Under apples-to-apples zero-shot, LookbackLens still trails LettuceDetect.** Best zero-shot is `ragtruth-w8-t0.7` at 0.270/0.073/0.166/0.326 — beats lexical floor by +3 pp on combined, but LettuceDetect-large (0.286/0.080/0.209/0.322) is consistently 1-5 pp ahead. The cross-domain transferability gap stands even after window and threshold tuning.

6. **Backbone size matters less than expected.** An earlier run with Qwen2.5-1.5B (28 × 12 = 336-dim features) gave essentially the same in-domain F1 as the current Llama-2-7B (32 × 32 = 1024-dim). Feature dim helped zero-shot by ~+2.5 pp but did not close the LettuceDetect gap — the bottleneck is cross-domain transferability of attention patterns, not dimensionality.

7. **LettuceDetect transfers better than LookbackLens.** Token-level NLI features in a fine-tuned encoder appear more portable than per-(layer × head) attention patterns from a frozen generator's forward pass — the attention signal carries the *generator*'s domain habits along with it, while LettuceDetect's encoder sees only the surface text.

8. **Contradiction is still the hardest type for every method** — single-token value swaps sit below the resolution of both attention-flow and token-NLI signals. Best F1 is 0.147 (`toolace-w4-t0.7`) vs LettuceDetect-large's 0.080.